# Natural Language to Cypher Backend Prototype

This notebook is a backend-shaped prototype for a future Flask endpoint in `skynet-web`.

It does four things:

1. Connects to the local Protostar Neo4j graph.
2. Builds a compact schema context for view 2.
3. Translates a natural language question into read-only Cypher.
4. Validates and executes the generated query.

The functions are intentionally plain Python so they can move into a Flask service module later.

## Setup

Expected environment variables:

- `NEO4J_URI`, default `bolt://localhost:7687`
- `NEO4J_USERNAME`, default `neo4j`
- `NEO4J_PASSWORD`
- `OPENAI_API_KEY`
- `OPENAI_MODEL`, optional

If a package is missing in your notebook kernel, install it in that environment:

```bash
pip install neo4j python-dotenv openai
```

In [ ]:
import json
import os
import re
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / ".env").exists():
    REPO_ROOT = Path.cwd().parent

load_dotenv(REPO_ROOT / ".env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")

if not NEO4J_PASSWORD:
    raise RuntimeError("Set NEO4J_PASSWORD in .env or the notebook environment.")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None

print(f"Repo root: {REPO_ROOT}")
print(f"Neo4j URI: {NEO4J_URI}")
print(f"OpenAI model: {OPENAI_MODEL}")
print(f"OpenAI API key loaded: {bool(os.getenv('OPENAI_API_KEY'))}")

In [ ]:
def run_read_query(cypher: str, parameters: dict[str, Any] | None = None) -> list[dict[str, Any]]:
    """Run a read-only query and return JSON-friendly row dictionaries."""
    with driver.session() as session:
        result = session.execute_read(
            lambda tx: [record.data() for record in tx.run(cypher, parameters or {})]
        )
    return [json_safe(row) for row in result]


def json_safe(value: Any) -> Any:
    """Convert common Neo4j driver values into values Flask can jsonify."""
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if hasattr(value, "items"):
        return {str(k): json_safe(v) for k, v in dict(value).items()}
    if hasattr(value, "iso_format"):
        return value.iso_format()
    return value


run_read_query("MATCH (n) RETURN count(n) AS nodes")

## Schema Context

Start with view 2 because it uses stable labels and relationship names. View 1 has dynamic labels based on detection names/types, which is useful later but less reliable for a first natural-language query layer.

In [ ]:
SCHEMA_CONTEXT = """
You translate analyst questions about a Neo4j security alert graph into Cypher.

Use only this graph model unless the user explicitly asks for a schema discovery query:

View 2 graph pattern:
(:ENTITY {view: 2})-[:HAS_SEVERITY]->(:SEVERITY_CLUSTER {view: 2})
(:SEVERITY_CLUSTER {view: 2})-[:NAME_CLUSTER]->(:NAME_CLUSTER {view: 2})
(:NAME_CLUSTER {view: 2})-[:INCLUDES]->(:ALERT {view: 2})

Labels:
- ENTITY
- SEVERITY_CLUSTER
- NAME_CLUSTER
- ALERT

ALERT properties:
guid, name, timestamp, detection_type, category, mitre_tactic, entity,
entity_type, host_ip, source_ip, dest_ip, dest_port, dst_geo, username,
syscall_name, executable, process, message, proctitle, severity, view

ENTITY properties:
ip, entity, entity_type, count, view

SEVERITY_CLUSTER properties:
ip, entity, entity_type, severity, count, view

NAME_CLUSTER properties:
ip, entity, entity_type, severity, name, count, view

Rules:
- Generate read-only Cypher only.
- Default to view = 2.
- Prefer ALERT for alert details.
- Prefer compact tabular RETURN fields over returning whole paths.
- Use case-insensitive comparisons with toLower(...) when matching user-provided text.
- Always include LIMIT 100 or lower.
- Do not use CREATE, MERGE, DELETE, DETACH, SET, REMOVE, DROP, LOAD CSV, CALL dbms, CALL apoc, or schema writes.
""".strip()

print(SCHEMA_CONTEXT)

In [ ]:
def sample_graph_values() -> dict[str, list[dict[str, Any]]]:
    """Small value samples help prompt tuning and UI affordances."""
    queries = {
        "severities": "MATCH (a:ALERT {view: 2}) RETURN a.severity AS severity, count(*) AS count ORDER BY count DESC LIMIT 20",
        "names": "MATCH (a:ALERT {view: 2}) RETURN a.name AS name, count(*) AS count ORDER BY count DESC LIMIT 20",
        "detection_types": "MATCH (a:ALERT {view: 2}) RETURN a.detection_type AS detection_type, count(*) AS count ORDER BY count DESC LIMIT 20",
        "entity_types": "MATCH (a:ALERT {view: 2}) RETURN a.entity_type AS entity_type, count(*) AS count ORDER BY count DESC LIMIT 20",
    }
    return {name: run_read_query(query) for name, query in queries.items()}


samples = sample_graph_values()
samples

## Cypher Generation

The model returns a strict JSON object so the future Flask route does not need to scrape prose. The generated query still goes through local validation before execution.

In [ ]:
CYPHER_PLAN_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "cypher": {"type": "string"},
        "explanation": {"type": "string"},
        "assumptions": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["cypher", "explanation", "assumptions"],
}


def get_openai_client() -> OpenAI:
    """Create the OpenAI client lazily so notebook cell order is less fragile."""
    existing_client = globals().get("client")
    if existing_client is not None:
        return existing_client

    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY in .env, then rerun the setup cell or this cell.")

    globals()["client"] = OpenAI()
    return globals()["client"]


def generate_cypher(question: str, samples: dict[str, list[dict[str, Any]]] | None = None) -> dict[str, Any]:
    prompt = {
        "question": question,
        "schema": SCHEMA_CONTEXT,
        "value_samples": samples or {},
    }
    response = get_openai_client().responses.create(
        model=globals().get("OPENAI_MODEL", os.getenv("OPENAI_MODEL", "gpt-5.5")),
        input=[
            {
                "role": "system",
                "content": "Return only a JSON object that satisfies the supplied schema.",
            },
            {"role": "user", "content": json.dumps(prompt)},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "cypher_plan",
                "schema": CYPHER_PLAN_SCHEMA,
                "strict": True,
            }
        },
    )
    return json.loads(response.output_text)


plan = generate_cypher("show me the 25 most recent high severity alerts", samples)
plan

## Guardrails

This is deliberately strict. The backend should fail closed: if a query looks unusual, reject it and ask the model or user to produce a simpler read-only query.

In [ ]:
FORBIDDEN_PATTERNS = [
    r"\bCREATE\b",
    r"\bMERGE\b",
    r"\bDELETE\b",
    r"\bDETACH\b",
    r"\bSET\b",
    r"\bREMOVE\b",
    r"\bDROP\b",
    r"\bLOAD\s+CSV\b",
    r"\bCALL\s+DBMS\b",
    r"\bCALL\s+APOC\b",
    r"\bCREATE\s+CONSTRAINT\b",
    r"\bCREATE\s+INDEX\b",
]


def validate_readonly_cypher(cypher: str) -> str:
    normalized = cypher.strip()
    upper = normalized.upper()

    if not normalized:
        raise ValueError("Cypher query is empty.")
    if ";" in normalized:
        raise ValueError("Semicolons are blocked to avoid multi-statement queries.")
    if "//" in normalized or "/*" in normalized or "*/" in normalized:
        raise ValueError("Comments are blocked in generated Cypher.")
    if not re.match(r"^\s*(MATCH|WITH|UNWIND)\b", normalized, flags=re.IGNORECASE):
        raise ValueError("Query must start with MATCH, WITH, or UNWIND.")
    if " RETURN " not in f" {upper} ":
        raise ValueError("Query must contain RETURN.")
    for pattern in FORBIDDEN_PATTERNS:
        if re.search(pattern, upper):
            raise ValueError(f"Blocked non-read-only Cypher pattern: {pattern}")

    limit_match = re.search(r"\bLIMIT\s+(\d+)\b", upper)
    if limit_match is None:
        normalized = f"{normalized}\nLIMIT 100"
    elif int(limit_match.group(1)) > 100:
        normalized = re.sub(r"\bLIMIT\s+\d+\b", "LIMIT 100", normalized, flags=re.IGNORECASE)

    return normalized


safe_cypher = validate_readonly_cypher(plan["cypher"])
print(safe_cypher)

In [ ]:
def explain_query(cypher: str) -> None:
    """Ask Neo4j to parse/plan the query before running it."""
    with driver.session() as session:
        session.run(f"EXPLAIN {cypher}").consume()


def ask_graph(question: str) -> dict[str, Any]:
    plan = generate_cypher(question, samples)
    cypher = validate_readonly_cypher(plan["cypher"])
    explain_query(cypher)
    rows = run_read_query(cypher)
    return {
        "question": question,
        "cypher": cypher,
        "rows": rows,
        "row_count": len(rows),
        "explanation": plan["explanation"],
        "assumptions": plan["assumptions"],
    }


answer = ask_graph("What are the top alert names by count?")
answer

## Flask Service Extraction Sketch

The notebook functions above can become a Flask service roughly like this:

```python
@app.post('/api/graph/search')
def graph_search():
    body = request.get_json(force=True)
    question = body.get('question', '').strip()
    if not question:
        return jsonify({'error': 'question is required'}), 400
    try:
        return jsonify(ask_graph(question))
    except ValueError as exc:
        return jsonify({'error': str(exc)}), 400
    except Exception as exc:
        current_app.logger.exception('graph search failed')
        return jsonify({'error': 'graph search failed'}), 500
```

Before production, use a read-only Neo4j user for this endpoint.

In [ ]:
# Close this when done with the notebook kernel.
# driver.close()